### Step 1: Import Libraries

In [0]:
from pyspark.sql import Row
from datetime import datetime
import time
import uuid


### Step 2: Create Database

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS medallion_db;

### Step 3: Create Execution Log Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS medallion_db.execution_logs
(
Batch_ID STRING,
Notebook_Name STRING,
Source_Layer STRING,
Target_Layer STRING,
Start_Time TIMESTAMP,
End_Time TIMESTAMP,
Duration_Seconds BIGINT,
Status STRING,
Records_Read BIGINT,
Records_Written BIGINT,
Error_Message STRING
)
USING DELTA;

In [0]:
display(spark.sql("SHOW TABLES IN medallion_db"))

database,tableName,isTemporary
medallion_db,execution_logs,false


### Step 4: Start Logger Function

In [0]:
def log_start(batch_id,
              notebook_name,
              source_layer,
              target_layer):

    start_time = datetime.now()

    return start_time

### Step 5: Success Logger Function

In [0]:
def log_success(batch_id,
                notebook_name,
                source_layer,
                target_layer,
                start_time,
                records_read,
                records_written):

    end_time = datetime.now()

    duration = int((end_time-start_time).total_seconds())

    log_df = spark.createDataFrame([(

        batch_id,
        notebook_name,
        source_layer,
        target_layer,
        start_time,
        end_time,
        duration,
        "SUCCESS",
        records_read,
        records_written,
        ""

    )],[
        "Batch_ID",
        "Notebook_Name",
        "Source_Layer",
        "Target_Layer",
        "Start_Time",
        "End_Time",
        "Duration_Seconds",
        "Status",
        "Records_Read",
        "Records_Written",
        "Error_Message"
    ])

    log_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("medallion_db.execution_logs")

### Step 6: Failure Logger Function

In [0]:
def log_failure(batch_id,
                notebook_name,
                source_layer,
                target_layer,
                start_time,
                records_read,
                records_written,
                error_message):

    end_time = datetime.now()

    duration = int((end_time-start_time).total_seconds())

    log_df = spark.createDataFrame([(

        batch_id,
        notebook_name,
        source_layer,
        target_layer,
        start_time,
        end_time,
        duration,
        "FAILED",
        records_read,
        records_written,
        error_message

    )],[
        "Batch_ID",
        "Notebook_Name",
        "Source_Layer",
        "Target_Layer",
        "Start_Time",
        "End_Time",
        "Duration_Seconds",
        "Status",
        "Records_Read",
        "Records_Written",
        "Error_Message"
    ])

    log_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("medallion_db.execution_logs")

### Step 7: View Execution Logs

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM medallion_db.execution_logs
    ORDER BY Start_Time DESC
    """)
)

In [0]:
# display(
#     spark.sql("""
#     TRUNCATE TABLE medallion_db.execution_logs
    
#     """)
# )

### 

In [0]:
spark.conf.set(
    "fs.azure.account.key.insurancedatahomecredit.blob.core.windows.net",
    "mvIlwSRtmT+AXedlxE40hoBJZsEQlvxnlHDD1ZdWje1R7m00TeP/QXcK1Gr7Nu8os0Mru1pMnb+E+AStH5S9mA=="
)

In [0]:
bronze_df = spark.read \
    .format("delta") \
    .load("wasbs://raw@insurancedatahomecredit.blob.core.windows.net/bronze/train")

In [0]:
records_read = bronze_df.count()